# Latent SOPE — 50 Rollouts

Scaled-up version of `latent_sope.ipynb` (Steps 0–3) with production-scale data collection.

| Parameter | `latent_sope.ipynb` | This notebook |
|-----------|--------------------|--------------|
| Oracle rollouts (K) | 10 | 50 |
| Offline rollouts (N) | 5 | 50 |
| Training epochs | 5 | 50 |
| Diffusion steps | 64 | 256 |

**Estimated runtime:** ~20 min total (Step 0: ~12 min, Step 1: ~12 min, Step 2: ~5 sec, Step 3: ~5 min).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import torch

REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Repo root: {REPO_ROOT}")

## Step 0: Ground Truth (Oracle Baseline)

50 rollouts for a tighter oracle estimate (~12 min).

In [ ]:
from src.latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint,
    build_rollout_policy_from_checkpoint,
    build_env_from_checkpoint,
)
from src.latent_sope.robomimic_interface.rollout import rollout

POLICY_DIR = Path("../third_party/robomimic/diffusion_policy_trained_models/test")
policy_train_dirs = sorted([d for d in POLICY_DIR.glob("*") if d.is_dir()])
assert len(policy_train_dirs) > 0, "No trained policies found."
policy_train_dir = policy_train_dirs[-1]
print(f"Using policy from: {policy_train_dir}")

ckpt = load_checkpoint(policy_train_dir.resolve(), ckpt_path="last.pth")
policy = build_rollout_policy_from_checkpoint(ckpt, device=torch.device(device), verbose=False)
env = build_env_from_checkpoint(ckpt, render=False, render_offscreen=True, verbose=False)

K = 50
HORIZON = 60
gamma = 1.0

oracle_returns = []
for i in range(K):
    stats = rollout(policy=policy, env=env, horizon=HORIZON, render=False)
    oracle_returns.append(stats.total_reward)
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{K}] running mean = {np.mean(oracle_returns):.3f}")

oracle_value = np.mean(oracle_returns)
print(f"\nOracle V^pi over {K} rollouts (horizon={HORIZON}, gamma={gamma}):")
print(f"  mean = {oracle_value:.3f}, std = {np.std(oracle_returns):.3f}")

## Step 1: Collect Offline Data

50 rollouts (~12 min). Saves to `rollout_latents_50/`.

In [ ]:
import h5py

from src.latent_sope.robomimic_interface.checkpoints import build_algo_from_checkpoint
from src.latent_sope.robomimic_interface.rollout import (
    PolicyFeatureHook,
    RolloutLatentRecorder,
    save_rollout_latents,
    load_rollout_latents,
    get_policy_frame_stack,
)

dataset_path = Path("../third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5")
algo = build_algo_from_checkpoint(ckpt)
global_cfg_keys = set(algo.global_config.all_obs_keys)

with h5py.File(dataset_path, "r") as h5:
    demo_keys = sorted(list(h5["data"].keys()))
    all_obs_keys = sorted(list(h5["data"][demo_keys[0]]["obs"].keys()))
    obs_keys = [k for k in all_obs_keys if k in global_cfg_keys]

print(f"Obs keys: {obs_keys}")

N_ROLLOUTS = 50
output_dir = policy_train_dir / "rollout_latents_50"
output_dir.mkdir(exist_ok=True)

frame_stack = get_policy_frame_stack(policy)

for i in range(N_ROLLOUTS):
    feature_hook = PolicyFeatureHook(policy, feat_type="low_dim_concat")
    recorder = RolloutLatentRecorder(
        feature_hook,
        obs_keys=obs_keys,
        store_obs=True,
        store_next_obs=False,
    )

    stats = rollout(policy=policy, env=env, horizon=HORIZON, render=False, recorder=recorder)
    traj = recorder.finalize(stats)

    save_path = output_dir / f"rollout_{i:03d}.h5"
    save_rollout_latents(save_path, traj)

    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{N_ROLLOUTS}] reward={stats.total_reward:.1f}, "
              f"latents={traj.latents.shape}")

    feature_hook.close()

rollout_paths = sorted(output_dir.glob("*.h5"))
print(f"\nCollected {len(rollout_paths)} rollout files in {output_dir}")

## Step 2: Chunk the Offline Data

50 rollouts x ~25 chunks each = ~1250 chunks → ~19 batches of 64.

In [ ]:
from src.latent_sope.robomimic_interface.dataset import (
    RolloutChunkDatasetConfig,
    make_rollout_chunk_dataloader,
)

sample_traj = load_rollout_latents(rollout_paths[0])
latents_dim = sample_traj.latents.shape[-1]
action_dim = sample_traj.actions.shape[-1]
print(f"Latent dim: {latents_dim}, Action dim: {action_dim}")
print(f"Latents shape: {sample_traj.latents.shape}")

dataset_config = RolloutChunkDatasetConfig(
    chunk_size=8,
    stride=2,
    frame_stack=2,
    source="latents",
    latents_dim=latents_dim,
    action_dim=action_dim,
    normalize=True,
    return_metadata=True,
)

BATCH_SIZE = 64

dataloader, norm_stats = make_rollout_chunk_dataloader(
    paths=rollout_paths,
    config=dataset_config,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

print(f"\nDataLoader: {len(dataloader)} batches of size {BATCH_SIZE}")
if norm_stats is not None:
    print(f"Normalization stats: mean shape={norm_stats.mean.shape}, std shape={norm_stats.std.shape}")

batch = next(iter(dataloader))
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape} {v.dtype}")
    else:
        print(f"  {k}: {type(v).__name__}")

## Step 3: Train Chunk Diffusion

256 diffusion steps (SOPE default), 50 epochs. ~5 min estimated.

In [ ]:
from tqdm.auto import tqdm
from src.latent_sope.diffusion.sope_diffuser import (
    SopeDiffusionConfig,
    SopeDiffuser,
    NormalizationStats as DiffusionNormStats,
    cross_validate_configs,
)

diffusion_config = SopeDiffusionConfig(
    chunk_horizon=dataset_config.chunk_size,
    frame_stack=dataset_config.frame_stack,
    state_dim=latents_dim,
    action_dim=action_dim,
    diffusion_steps=256,
    dim_mults=(1, 2),
    attention=False,
    loss_type="l2",
    action_weight=5.0,
    predict_epsilon=True,
    lr=3e-4,
    guided=False,
)

cross_validate_configs(dataset_config, diffusion_config)
print("Config cross-validation passed.")

diff_norm_stats = None
if norm_stats is not None:
    diff_norm_stats = DiffusionNormStats(
        mean=norm_stats.mean,
        std=norm_stats.std,
    )

diffuser = SopeDiffuser(
    cfg=diffusion_config,
    normalization_stats=diff_norm_stats,
    device=device,
)
optimizer = diffuser.make_optimizer()

n_params = sum(p.numel() for p in diffuser.diffusion.parameters())
print(f"TemporalUnet parameters: {n_params:,}")
print(f"Chunk: {diffusion_config.total_chunk_horizon} steps "
      f"({diffusion_config.frame_stack} context + {diffusion_config.chunk_horizon} target)")
print(f"Transition dim: {diffuser.transition_dim} "
      f"({diffuser.state_dim} state + {diffuser.action_dim} action)")

In [ ]:
import matplotlib.pyplot as plt

EPOCHS = 50
GRAD_CLIP = 1.0

losses = []
diffuser.diffusion.train()

for epoch in range(1, EPOCHS + 1):
    epoch_losses = []
    for batch in tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        batch_dev = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }

        loss, info = diffuser.loss(batch_dev)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(diffuser.diffusion.parameters(), GRAD_CLIP)
        optimizer.step()

        losses.append(loss.item())
        epoch_losses.append(loss.item())

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}: mean loss = {np.mean(epoch_losses):.4f}")

plt.figure(figsize=(10, 3))
plt.plot(losses, alpha=0.3, color="steelblue")
window = min(50, max(1, len(losses) // 5))
if window > 1:
    smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
    plt.plot(range(window - 1, len(losses)), smoothed, color="steelblue", linewidth=2)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title(f"Chunk diffusion training loss ({EPOCHS} epochs, {len(rollout_paths)} rollouts)")
plt.tight_layout()
plt.show()

In [ ]:
from dataclasses import asdict

ckpt_dir = policy_train_dir / "diffusion_ckpts_50"
ckpt_dir.mkdir(exist_ok=True)

ckpt_payload = {
    "diffusion_state_dict": diffuser.diffusion.state_dict(),
    "epoch": EPOCHS,
    "step": len(losses),
    "diffusion_config": asdict(diffusion_config),
    "dataset_config": asdict(dataset_config),
    "normalization_stats": {
        "mean": norm_stats.mean,
        "std": norm_stats.std,
    } if norm_stats is not None else None,
}

ckpt_path = ckpt_dir / "sope_diffuser_latest.pt"
torch.save(ckpt_payload, str(ckpt_path))
print(f"Saved diffusion checkpoint to {ckpt_path}")
print(f"  Epochs: {EPOCHS}, Steps: {len(losses)}, Final loss: {losses[-1]:.4f}")
print(f"  Oracle V^pi: {oracle_value:.3f} (from {K} rollouts)")